# Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0 xformers==0.0.32.post2

# Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch


# We can try testing granite 4 small on a big GPU if micro isn't that good
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 1024,   # Same as inference script
    load_in_4bit = False,    # Load full precision for max accuracy
    load_in_8bit = False,
    full_finetuning = False, # Don't do this, LoRA gets just as good results
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

Side note, QLoRA is not recommended for **any** Qwen 3.5 model due to higher than normal reduction in performance from quantization.

Base template from Unsloth: r = 8, alpha = 16

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "out_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # Just go with Unsloth recommendation
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


# Data Prep

The chat template for qwen look like this:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```

Need to format the data into the following conversational style:

```
{"role": "system", "content": "system prompt"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [4]:
# From Sakhawat et al., 2026

default_system_prompt = """
You are participating in a standardized News Bias Classification task for academic
research. Output ONLY a single numeric value between -3.0 and +3.0.

Do NOT provide explanations or text.
"""

In [5]:
# Load the dataset
from datasets import load_dataset

train_dataset = load_dataset('avanishd/ground-news-2026', split='train')
validation_dataset = load_dataset('avanishd/ground-news-2026', split='validation')

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/395k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/375k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8642 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1884 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1836 [00:00<?, ? examples/s]

In [6]:
def formatting_prompts_func(examples):

    bias_mapping = {
        "Far Left": "-3.0",
        "Left": "-2.0",
        "Lean Left": "-1.0",
        "Center": "0.0",
        "Lean Right": "+1.0",
        "Right": "+2.0",
        "Far Right": "+3.0"
    }

    messages = [
        [
            {"role": "system", "content": default_system_prompt},
            {"role": "user", "content": f"""
            HEADLINE: {headline}
            ARTICLE SUMMARY: {summary}

            Output ONLY the numeric bias score.
            """},
            {"role": "assistant", "content": bias_mapping[true_bias]}
        ] for headline, summary, true_bias in zip(examples['headline'], examples['summary'], examples['bias'])
    ]

    texts = [tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt = False) for message in messages]

    return { "text" : texts, }

train_dataset = train_dataset.map(formatting_prompts_func, batched = True,)
validation_dataset = validation_dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/8642 [00:00<?, ? examples/s]

Map:   0%|          | 0/1836 [00:00<?, ? examples/s]

Look at how the chat template mapped the conversation

In [7]:
train_dataset[5]["text"]

"<|im_start|>system\nYou are participating in a standardized News Bias Classification task for academic\nresearch. Output ONLY a single numeric value between -3.0 and +3.0.\n\nDo NOT provide explanations or text.<|im_end|>\n<|im_start|>user\nHEADLINE: Three New Rulings, One Goes President Trump's Way\n            ARTICLE SUMMARY: Three new rulings and one unexpected victory for Donald Trump. Pursuing Chief Lyons Chief Judge Patrick Schiltz of Minnesota’s federal court ordered the acting chief of U.S. Immigration and Customs Enforcement, Todd Lyons, to appear in court on Friday. He must personally explain why that agency has not complied with a slew of court orders. […] The post Three New Rulings, One Goes President Trump’s Way appeared first on www.independentsentinel.co…\n\n            Output ONLY the numeric bias score.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n+2.0<|im_end|>\n"

# Training


Effective batch size = per_device_train_batch_size * gradient_accumulation steps

With Unsloth, per_device_train_batch_size and gradient_accumulation steps are equivalent, which may not be the case in other frameworks.

This setup (with effective batch size 32) is the fastest one. Putting gradient accumulation steps to 1, 2, or 8 (and modifying per device train batch size to maintain effective batch size) is slower.

In [8]:
from trl import SFTTrainer, SFTConfig

# 3 evals per epoch
PER_DEVICE_TRAIN_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1

EFFECTIVE_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

EVAL_STEPS = len(train_dataset) // EFFECTIVE_BATCH_SIZE // 3

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 1, # Increasing "might" throw error on Colab/other envs.
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS, # Use GA to mimic batch size!
        warmup_ratio = 0.05,
        num_train_epochs = 1, # Full training runs, can experiment w/ multiple
 #       max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = EVAL_STEPS,
        per_device_eval_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE, # Using higher batch size to speed up eval
        eval_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc (TODO: Set up when dataset is finalized)
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [9]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n<think>",
)

Map (num_proc=16):   0%|          | 0/8642 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/8642 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/8642 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/1836 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/1836 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1836 [00:00<?, ? examples/s]

Verify masking the instruction part is done

In [10]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'<|im_start|>system\nYou are participating in a standardized News Bias Classification task for academic\nresearch. Output ONLY a single numeric value between -3.0 and +3.0.\n\nDo NOT provide explanations or text.<|im_end|>\n<|im_start|>user\nHEADLINE: Trump picks Kevin Warsh for Fed chair, but key Republican vows to block him over Powell investigation\n            ARTICLE SUMMARY: President Donald Trump announced conservative economist and former Fed governor Kevin Warsh as his pick to be the new Federal Reserve chairman.\n\n            Output ONLY the numeric bias score.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n-1.0<|im_end|>\n'

Print out the masked out example - should only see the assistant response

In [11]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                  \n\n</think>\n\n-1.0<|im_end|>\n'

In [12]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
8.559 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [13]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,642 | Num Epochs = 1 | Total steps = 271
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 23,789,568 of 4,563,055,104 (0.52% trained)


Step,Training Loss,Validation Loss
90,0.154045,0.166280
180,0.106373,0.106179
270,0.078417,0.094346


In [14]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

606.5863 seconds used for training.
10.11 minutes used for training.
Peak reserved memory = 33.707 GB.
Peak reserved memory for training = 25.148 GB.
Peak reserved memory % of max memory = 85.347 %.
Peak reserved memory for training % of max memory = 63.675 %.


# Saving model

In [15]:
from google.colab import userdata
model.push_to_hub_merged("avanishd/Qwen3.5-4B-Ground-News", tokenizer, save_method = "merged_16bit", token = userdata.get('HF_TOKEN'))


No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...round-News/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

No files have been modified since last commit. Skipping to prevent empty commit.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   1%|1         | 47.9MB / 3.99GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:37<00:00, 78.68s/it]


Unsloth: Merge process complete. Saved to `/content/avanishd/Qwen3.5-4B-Ground-News`
